# レッスン11: 実務の作法 — 「動くコード」から「良いコード」へ ✨

文法はもう一通り知っています。ここからは**プロの現場で「読みやすい」と言われるコードの書き方**。実務ではコードは必ず他人(と3ヶ月後の自分)に読まれ、**コードレビュー**で指摘し合う文化があります。今日はレビューで指摘されない書き方を身につけます。

---
## 11-1. PEP 8 — Python公式スタイルガイド

Pythonには公式の「書き方の決まり」PEP 8 があり、世界中の現場がこれに従います。最重要ポイント:

| ルール | 悪い例 | 良い例 |
|--------|--------|--------|
| 変数・関数は snake_case | `dtMax`, `DtMax` | `dt_max` |
| クラスは CamelCase | `bank_account` | `BankAccount` |
| 定数は全大文字 | `gravity = 9.8` | `GRAVITY = 9.8` |
| `=` や演算子の前後にスペース | `a=b+1` | `a = b + 1` |
| ただし引数のデフォルト値は詰める | `f(x = 1)` | `f(x=1)` |
| 1行は79〜99文字まで | 横に無限に長い行 | 適度に改行 |

丸暗記は不要です。実務では**自動整形ツール(formatter)**が直してくれるので、「そういうルールがある」と知っていることが大事。

---
## 11-2. 名前は「嘘をつかない・省略しない」

コードは書く時間より読まれる時間のほうが長い。だから名前が9割です。

In [ ]:
# ❌ 読めないコード(動きはする)
def f(l, t):
    r = []
    for x in l:
        if x > t:
            r.append(x)
    return r

# ✅ 同じ処理。名前を付け直しただけで、コメント無しでも意図が伝わる
def filter_above_threshold(values, threshold):
    result = []
    for value in values:
        if value > threshold:
            result.append(value)
    return result

print(filter_above_threshold([1, 8, 3, 9, 2], threshold=5))

実務の命名の合言葉:

- **省略しない**: `tmp`, `res`, `d` より `temporary_file`, `result`, `date`
- **単位や意味を含める**: `wait` より `wait_seconds`、`size` より `size_mb`
- **boolはis/hasで始める**: `flag` より `is_stable`, `has_error`

---
## 11-3. マジックナンバーを消す

コード中に突然現れる裸の数字(マジックナンバー)は、意味も変更箇所も分からなくなる実務の地雷です。

In [ ]:
# ❌ 3600って何?0.5って何?
def check(dt, dx, a):
    if a * dt / dx**2 > 0.5:
        print("NG")

# ✅ 名前をつけて意図を宣言する。定数は大文字(PEP 8)
STABILITY_LIMIT = 0.5          # 陽解法の安定条件の上限
SECONDS_PER_HOUR = 3600

def check_stability(dt, dx, a):
    r = a * dt / dx ** 2
    if r > STABILITY_LIMIT:
        print(f"NG: r = {r:.3f} が上限 {STABILITY_LIMIT} を超えています")
    else:
        print(f"OK: r = {r:.3f}")

check_stability(dt=20, dx=0.01, a=8.264e-7)
check_stability(dt=100, dx=0.01, a=8.264e-7)

---
## 11-4. docstringと型ヒント — 関数の「取扱説明書」

実務の関数には**docstring**(説明文)と**型ヒント**(引数・戻り値の型の注記)を付けます。エディタが補完・警告してくれるようになり、チーム開発の事故が激減します。

In [ ]:
def thermal_diffusivity(lam: float, rho: float, c: float) -> float:
    """熱拡散率 a = λ/(ρc) を計算する。

    Args:
        lam: 熱伝導率 [W/(m·K)]
        rho: 密度 [kg/m³]
        c:   比熱 [J/(kg·K)]

    Returns:
        熱拡散率 [m²/s]
    """
    return lam / (rho * c)

print(thermal_diffusivity(1.6, 2200, 880))
help(thermal_diffusivity)      # docstringはhelp()やエディタのポップアップに表示される

`lam: float` が型ヒント、`-> float` が「floatを返す」の意味。**Python自体は型ヒントを強制しません**(書かなくても動く)が、実務の新規コードではほぼ標準になっています。

---
## 11-5. プログラム全体の標準構造

実務の.pyファイルには定番のレイアウトがあります:

```python
"""ファイル全体の説明(モジュールdocstring)"""

# 1. import(標準 → 外部 → 自作の順)
import math
import numpy as np

# 2. 定数
STABILITY_LIMIT = 0.5

# 3. 関数・クラスの定義
def main():
    """全体の流れはmain関数にまとめる"""
    ...

# 4. 実行の入口
if __name__ == "__main__":
    main()
```

最後の `if __name__ == "__main__":` は「このファイルが**直接実行されたときだけ** main() を動かす(importされたときは動かさない)」というPythonの定型句。レッスン9のモジュール分割と組み合わせるための仕組みです。呪文に見えますが実務コードの9割に書いてあるので、意味を知っておくと「読める」度が上がります。

---
## ✏️ 練習問題 11-A 【コードレビュー体験】

次の「動くけどひどい」コードを、今日の作法でリファクタリング(動作を変えずに書き直し)してください:

```python
def calc(x,y,z):
    if z==1:
        return x*y*0.1
    if z==2:
        return x*y*0.08
```

ヒント: これは「商品単価x・個数yから、z=1なら税率10%、z=2なら軽減税率8%の消費税額を計算する」関数だそうです。名前・定数・型ヒント・docstring・スペースを直しましょう。zが1でも2でもないときどうするかも考えられたら完璧(レッスン8!)。

In [ ]:
# ここにリファクタリング後のコードを書いてください


---
## ✏️ 練習問題 11-B 【main構造】

レッスン6の熱伝導コード(お手本)を、実務レイアウトに再構成してください:

- `setup()` … 条件を辞書で返す関数
- `simulate(config)` … 計算して結果を返す関数
- `plot_result(...)` … グラフを描く関数
- `main()` … 上の3つを順に呼ぶ
- 最後に `if __name__ == "__main__": main()`

分割の仕方に唯一の正解はありません。「1関数1仕事」になっていればOK。これが書ければ、**あなたのコードはもう実務の見た目**です。

In [ ]:
# ここにコードを書いてください


---
## 🎉 レッスン11はここまで!

**今日覚えたこと:**
1. PEP 8: snake_case / CamelCase / 大文字定数 / スペースの入れ方
2. 名前は省略せず、単位と意味を含め、boolはis/has
3. マジックナンバーは名前つき定数に
4. docstring + 型ヒント = 関数の取説
5. `if __name__ == "__main__":` を含む標準レイアウト

**次回 → レッスン12: Git/GitHub**(チーム開発の土台。最終回!)

---
### 💡 答え(一例)

<details>
<summary>クリックで表示</summary>

```python
# 11-A
TAX_RATE_STANDARD = 0.10   # 標準税率
TAX_RATE_REDUCED = 0.08    # 軽減税率

def calc_tax(unit_price: float, quantity: int, tax_type: int) -> float:
    """消費税額を計算する。

    Args:
        unit_price: 商品単価 [円]
        quantity:   個数
        tax_type:   1 = 標準税率, 2 = 軽減税率

    Returns:
        消費税額 [円]
    """
    subtotal = unit_price * quantity
    if tax_type == 1:
        return subtotal * TAX_RATE_STANDARD
    if tax_type == 2:
        return subtotal * TAX_RATE_REDUCED
    raise ValueError(f"不明な税区分です: {tax_type}")

# 11-B (骨格の一例)
import numpy as np
import matplotlib.pyplot as plt

RECORD_TIMES = {180: "3 min", 1800: "30 min", 7200: "2 h",
                14400: "4 h", 28800: "8 h", 43200: "12 h"}

def setup() -> dict:
    """計算条件を返す"""
    lam, rho, c = 1.6, 2200, 880
    return {"L": 0.3, "n": 30, "dt": 20, "a": lam / (rho * c)}

def simulate(config: dict) -> tuple:
    """温度分布の時間発展を計算し、(x, {時刻ラベル: 分布}) を返す"""
    n, dt, a = config["n"], config["dt"], config["a"]
    dx = config["L"] / n
    r = a * dt / dx ** 2
    x = np.linspace(0, config["L"], n + 1)
    T = np.zeros(n + 1)
    T[0] = T[-1] = 1.0
    snapshots = {}
    for step in range(1, int(43200 / dt) + 1):
        T_new = T.copy()
        T_new[1:-1] = T[1:-1] + r * (T[:-2] - 2*T[1:-1] + T[2:])
        T = T_new
        if step * dt in RECORD_TIMES:
            snapshots[RECORD_TIMES[step * dt]] = T.copy()
    return x, snapshots

def plot_result(x, snapshots):
    """時刻ごとの温度分布を1枚のグラフに描く"""
    for label, T in snapshots.items():
        plt.plot(x * 1000, T, label=label)
    plt.xlabel("x [mm]")
    plt.ylabel("T [C]")
    plt.legend()
    plt.grid(True)
    plt.show()

def main():
    config = setup()
    x, snapshots = simulate(config)
    plot_result(x, snapshots)

if __name__ == "__main__":
    main()
```
</details>